# FisheriesAudit ALG — Entrega #09
# Geovisor SERE (INIDEP): Vedas Geoespaciales como Ground Truth Externo

> **Autor:** Ariel Giamportone — Serie FisheriesAudit ALG 2026  
> **Fecha:** junio 2026  
> **Repositorio:** arielgiamportone/cfp-audit-intelligence

---

## Resumen ejecutivo

El INIDEP publicó **SERE — Visualizador de Especies** (`sere.inidep.edu.ar`), un geovisor
que corre sobre GeoServer y expone capas vía servicios OGC estándar (WFS 2.0.0), **públicos
y sin autenticación**. Las capas de **vedas geoespaciales 2024** citan directamente el
**número de resolución, el organismo emisor (CFP/CTMFM) y un link al PDF oficial** —
una fuente *externa al parser* que sirve como ground truth para auditar qué fracción de
las resoluciones de veda efectivamente vigentes aparece citada en el corpus de actas CFP.

Este notebook:
1. Descarga (o reutiliza) las zonas de veda del geovisor y las persiste en `vedas_geoespaciales`.
2. Cruza esas citas contra el corpus de actas vía `GeovisorCrossValidator` (ADR-009).
3. Documenta la **lección metodológica central**: CFP y CTMFM numeran sus resoluciones de
   forma independiente — comparar sin filtrar por organismo produce falsos positivos.

**Diseño completo:** `docs/adr/009-geovisor-sere-inidep.md`.

---


## 1. Setup e importaciones

In [ ]:
import sys
from pathlib import Path

# Agregar src al path si se ejecuta desde notebooks/
repo_root = Path('.').resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import matplotlib.pyplot as plt
import pandas as pd
import sqlite3

from src.acquisition.inidep_geovisor_scraper import SEREGeovisorClient, VEDAS_LAYERS
from src.analysis.geovisor_cross_validator import GeovisorCrossValidator

plt.style.use('dark_background')
pd.set_option('display.max_colwidth', 80)

DB_PATH = Path('../data/processed/catalog.db')
DB_PATH

## 2. Descargar (o reutilizar) las vedas geoespaciales del geovisor

`SEREGeovisorClient.scrape_and_save_vedas` consulta las 16 capas WFS `vedas_2024:*`
relevantes para las especies ya verificadas en el comparador (merluza, centolla, abadejo,
polaca, langostino, calamar illex, merluza negra, vieira, rincón) y persiste las zonas
nuevas en `vedas_geoespaciales` (deduplicado por `capa, area, resolucion_numero`).

> ⚠️ Requiere conexión a `sere.inidep.edu.ar`. Si ya se corrió `--step geovisor` del
> pipeline, esta celda no agrega filas nuevas (idempotente).

In [ ]:
client = SEREGeovisorClient(delay=1.0)
try:
    n_nuevas = client.scrape_and_save_vedas(DB_PATH)
    print(f'Zonas de veda nuevas persistidas: {n_nuevas}')
except Exception as exc:
    print(f'No se pudo consultar el geovisor en este entorno: {exc}')
    print('Continuamos con los datos ya persistidos en vedas_geoespaciales (si existen).')

with sqlite3.connect(DB_PATH) as conn:
    try:
        df_vedas = pd.read_sql_query('SELECT * FROM vedas_geoespaciales', conn)
    except Exception:
        df_vedas = pd.DataFrame()

print(f'Total de zonas de veda en la base: {len(df_vedas)}')
df_vedas.head(10)

### 2.1 Distribución de zonas de veda por especie

In [ ]:
if not df_vedas.empty:
    conteo = df_vedas.groupby('especie_code', dropna=True).size().sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(10, 5))
    conteo.plot(kind='barh', ax=ax, color='#00838F')
    ax.set_xlabel('N° de zonas de veda georreferenciadas')
    ax.set_title('Zonas de veda 2024 por especie (geovisor SERE / INIDEP)')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig('fig09_vedas_por_especie.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figura guardada: fig09_vedas_por_especie.png')
else:
    print('Sin datos de vedas geoespaciales — correr la celda de descarga con conexión a internet.')

## 3. Resoluciones citadas por el geovisor: número, organismo y link al PDF

Cada zona de veda trae el número de resolución y el link directo al PDF oficial —
verificable de forma independiente del parser propio del proyecto.

In [ ]:
if not df_vedas.empty:
    citadas = (
        df_vedas[df_vedas['resolucion_numero'].notna()]
        [['resolucion_numero', 'resolucion_fuente', 'resolucion_url', 'especie_code']]
        .drop_duplicates(subset=['resolucion_numero', 'resolucion_fuente'])
        .sort_values(['resolucion_fuente', 'resolucion_numero'])
        .reset_index(drop=True)
    )
    print(f'Resoluciones únicas citadas por el geovisor: {len(citadas)}')
    print('\nPor organismo emisor:')
    print(citadas['resolucion_fuente'].value_counts())
    display(citadas)
else:
    citadas = pd.DataFrame()
    print('Sin datos.')

## 4. Cruce de cobertura contra el corpus de actas (`GeovisorCrossValidator`)

### 4.1 La lección metodológica central (verificada empíricamente, ADR-009)

CFP y **CTMFM** (Comisión Técnica Mixta del Frente Marítimo, organismo binacional
Argentina–Uruguay) **numeran sus resoluciones de forma independiente**. Se verificó
en vivo que:

- El geovisor cita **"Res. 13/2024"** como una veda de condrictios emitida por
  **CTMFM** (`ctmfm.org/resoluciones/Res%2013%202024...`).
- El corpus de actas cita **"Resolución CFP N° 13/2024"** — una norma completamente
  distinta, de **CFP**, sobre la CMP de merluza común.

Mismo número, mismo año, **organismos y documentos distintos**. Una comparación ingenua
por `(número, año)` reportaría esto como "cobertura encontrada" — un falso positivo.

**Por eso `GeovisorCrossValidator.validar_cobertura()` filtra explícitamente por
`fuente == 'CFP'`** antes de buscar coincidencias en el corpus (que son actas del CFP).

In [ ]:
validador = GeovisorCrossValidator(DB_PATH)
resultados = validador.validar_cobertura()
resumen = validador.cobertura_summary(resultados)

print('=== COBERTURA DEL CORPUS — RESOLUCIONES DE VEDA CITADAS POR EL GEOVISOR (fuente=CFP) ===')
print(f"  Total citadas (CFP)      : {resumen['total_resoluciones_citadas_cfp']}")
print(f"  Encontradas en el corpus : {resumen['encontradas_en_corpus']}")
print(f"  Pendientes               : {resumen['pendientes']}")
print(f"  % cobertura              : {resumen['pct_cobertura']}%")
print(f"\n{resumen['interpretacion']}")

### 4.2 Detalle por resolución

In [ ]:
df_cobertura = pd.DataFrame([
    {
        'resolucion_numero': r.resolucion_numero,
        'especies': ', '.join(e for e in r.especies if e),
        'url': r.resolucion_url,
        'encontrada_en_corpus': '✅' if r.encontrada_en_corpus else '⏳ pendiente',
        'n_actas_que_la_citan': len(r.resolucion_ids_corpus),
    }
    for r in resultados
])
df_cobertura

## 5. Interpretación y ruta de migración

**Hallazgo actual:** la cobertura es baja (o nula) porque el corpus de actas todavía no
incluye, para los años citados por el geovisor (2018, 2019, 2024), resoluciones de
`tipo='veda'` con el mismo número — el mismo bloqueante documentado en ADR-008
(auditoría de citas INIDEP): **el pipeline real (`--step process`) aún no cargó esas actas**.

Esto **no invalida la métrica**: a diferencia de ADR-008, el origen de datos del geovisor
es independiente del propio parser — no depende del pipeline para *existir*, solo para
*cruzarse*. El validador corre hoy sobre el corpus existente y se vuelve más representativo
a medida que el corpus crece.

### Ruta de migración (ADR-009)

1. Correr `--step process` real → poblar `resoluciones` con las actas 2018/2019/2024.
2. Re-ejecutar este notebook (o `--step geovisor` del pipeline) → la cobertura debería subir.
3. Si la cobertura es alta de forma sostenida → usar el geovisor como **fuente de
   verificación automática continua** del parser (cada año el INIDEP publica `vedas_<año>`).

## 6. Tabla LaTeX para paper

Tabla de resoluciones de veda (fuente CFP) citadas por el geovisor INIDEP, lista para
incluir en el artículo de la Serie ALG.

In [ ]:
if not df_cobertura.empty:
    latex = df_cobertura[['resolucion_numero', 'especies', 'encontrada_en_corpus']].to_latex(
        index=False,
        caption='Resoluciones de veda (fuente CFP) citadas por el geovisor SERE del INIDEP '
                'y su cobertura en el corpus de actas cargado (ADR-009).',
        label='tab:geovisor_cobertura',
    )
    print(latex)
else:
    print('Sin datos para tabla LaTeX.')

## 7. Conclusiones y próximos pasos

### Hallazgos preliminares

1. **Nueva fuente externa verificable**: el geovisor SERE aporta una lista de
   resoluciones de veda con número, organismo y link al PDF — independiente del parser
   propio, reutilizable como ground truth para auditar la cobertura del corpus.

2. **Lección metodológica documentada**: la colisión de numeración CFP/CTMFM
   ("Res. 13/2024" existe en ambos organismos, refiriendo a normas distintas) obliga a
   filtrar por `fuente` en cualquier cruce — error que de no detectarse hubiera producido
   una métrica de cobertura inflada y engañosa.

3. **Cobertura actual limitada por el mismo bloqueante que ADR-008**: el corpus de actas
   no incluye aún las resoluciones de veda de los años citados (2018, 2019, 2024) — se
   resuelve corriendo el pipeline real (`--step process`).

### Próximos pasos

- Ejecutar `--step process` para los años 2018, 2019 y 2024 y re-medir la cobertura.
- Si la cobertura sostenida es alta, promover el geovisor a **fuente de verificación
  continua** (Entrega futura: alerta automática cuando una veda del geovisor no aparece
  citada en el corpus dentro de N meses de su publicación).
- Extender el cruce a las **195 capas de distribución de especies** (ocurrencias
  georreferenciadas desde 1982) para anclar geoespacialmente `inidep_evaluaciones.zona`
  y `cfp_cuotas.zona` (hoy texto libre).

---

*Serie FisheriesAudit ALG — Entrega #09. Repositorio: `arielgiamportone/cfp-audit-intelligence`.*